# E-commerce Customer Segmentation Analysis

This notebook demonstrates how to use the customer segmentation model to analyze e-commerce customer data and identify distinct customer groups for targeted marketing strategies.

## 1. Setup and Imports

In [ ]:
# Import required libraries
import sys
import os
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from customer_segmentation import CustomerSegmentationModel, find_optimal_clusters
from data_generator import generate_ecommerce_data, get_data_summary

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

## 2. Generate Sample E-commerce Data

In [ ]:
# Generate sample e-commerce transaction data
print("Generating sample e-commerce data...")
data = generate_ecommerce_data(
    n_customers=500,
    n_products=50,
    date_range_days=365,
    random_state=42
)

print(f"Generated {len(data):,} transactions for {data['customer_id'].nunique():,} customers")
print(f"Date range: {data['order_date'].min()} to {data['order_date'].max()}")

In [ ]:
# Display sample data
print("Sample transaction data:")
data.head(10)

## 3. Explore the Data

In [ ]:
# Get data summary
summary = get_data_summary(data)
print("Data Summary:")
for key, value in summary.items():
    if key != 'category_distribution':
        print(f"{key.replace('_', ' ').title()}: {value}")

print("\nCategory Distribution:")
for category, count in summary['category_distribution'].items():
    percentage = (count / summary['total_transactions']) * 100
    print(f"  {category}: {count:,} ({percentage:.1f}%)")

In [ ]:
# Visualize data distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Orders per customer
orders_per_customer = data.groupby('customer_id')['order_id'].nunique()
axes[0, 0].hist(orders_per_customer, bins=20, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Number of Orders')
axes[0, 0].set_ylabel('Number of Customers')
axes[0, 0].set_title('Distribution of Orders per Customer')

# Spending per customer
spending_per_customer = data.groupby('customer_id')['total_amount'].sum()
axes[0, 1].hist(spending_per_customer, bins=20, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Total Spending ($)')
axes[0, 1].set_ylabel('Number of Customers')
axes[0, 1].set_title('Distribution of Total Spending per Customer')

# Category distribution
category_counts = data['product_category'].value_counts()
axes[1, 0].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%')
axes[1, 0].set_title('Product Category Distribution')

# Order value distribution
order_values = data.groupby('order_id')['total_amount'].sum()
axes[1, 1].hist(order_values, bins=30, edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Order Value ($)')
axes[1, 1].set_ylabel('Number of Orders')
axes[1, 1].set_title('Distribution of Order Values')

plt.tight_layout()
plt.show()

## 4. Find Optimal Number of Clusters

In [ ]:
# Find optimal number of clusters using elbow method
print("Finding optimal number of clusters...")
optimal_k, wcss_values = find_optimal_clusters(data, max_clusters=8)

print(f"Optimal number of clusters: {optimal_k}")

# Plot elbow curve
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(wcss_values) + 1), wcss_values, marker='o', linewidth=2, markersize=8)
plt.axvline(x=optimal_k, color='red', linestyle='--', label=f'Optimal k = {optimal_k}')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.title('Elbow Method for Optimal Number of Clusters')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Build and Train Customer Segmentation Model

In [ ]:
# Create and train the segmentation model
model = CustomerSegmentationModel(n_clusters=optimal_k, random_state=42)
print(f"Training model with {optimal_k} clusters...")

# Fit the model
model.fit(data)
print("Model training completed!")

## 6. Evaluate Model Performance

In [ ]:
# Evaluate model performance
evaluation = model.evaluate_model(data)
print("Model Evaluation Metrics:")
for metric, value in evaluation.items():
    print(f"{metric.replace('_', ' ').title()}: {value}")

## 7. Analyze Customer Segments

In [ ]:
# Get cluster summary statistics
summary_stats = model.get_cluster_summary(data)
print("Segment Summary Statistics:")
print(summary_stats.round(2))

In [ ]:
# Generate business insights for each segment
insights = model.get_segment_insights(data)
customer_features = model.prepare_features(data)
customer_features['cluster'] = model.labels_

print("CUSTOMER SEGMENT INSIGHTS:")
print("=" * 50)

for cluster, insight in insights.items():
    cluster_size = len(customer_features[customer_features['cluster'] == cluster])
    percentage = (cluster_size / len(customer_features)) * 100
    
    print(f"\nSEGMENT {cluster} ({cluster_size} customers - {percentage:.1f}%)")
    print("-" * 40)
    print(f"Insight: {insight}")
    
    # Show key metrics for this segment
    cluster_data = customer_features[customer_features['cluster'] == cluster]
    print(f"Average Recency: {cluster_data['recency'].mean():.1f} days")
    print(f"Average Frequency: {cluster_data['frequency'].mean():.1f} orders")
    print(f"Average Monetary: ${cluster_data['monetary'].mean():.2f}")
    print(f"Average Order Value: ${cluster_data['avg_order_value'].mean():.2f}")

## 8. Visualize Customer Segments

In [ ]:
# Create comprehensive segment visualizations
model.plot_segments(data, figsize=(16, 12))
plt.show()

In [ ]:
# Additional detailed visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Segment size distribution
segment_sizes = customer_features['cluster'].value_counts().sort_index()
axes[0, 0].pie(segment_sizes.values, labels=[f'Segment {i}' for i in segment_sizes.index], 
               autopct='%1.1f%%', startangle=90)
axes[0, 0].set_title('Customer Distribution by Segment')

# RFM heatmap by segment
rfm_by_cluster = customer_features.groupby('cluster')[['recency', 'frequency', 'monetary']].mean()
sns.heatmap(rfm_by_cluster.T, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[0, 1])
axes[0, 1].set_title('Average RFM Values by Segment')
axes[0, 1].set_ylabel('RFM Metrics')

# Segment comparison - violin plots
axes[1, 0].violinplot([customer_features[customer_features['cluster'] == i]['monetary'].values 
                      for i in range(optimal_k)])
axes[1, 0].set_xlabel('Segment')
axes[1, 0].set_ylabel('Monetary Value')
axes[1, 0].set_title('Monetary Value Distribution by Segment')
axes[1, 0].set_xticks(range(1, optimal_k + 1))
axes[1, 0].set_xticklabels([f'Segment {i}' for i in range(optimal_k)])

# Average order value by segment
avg_values = customer_features.groupby('cluster')['avg_order_value'].mean()
bars = axes[1, 1].bar(range(len(avg_values)), avg_values.values, 
                     color=plt.cm.viridis(np.linspace(0, 1, len(avg_values))))
axes[1, 1].set_xlabel('Segment')
axes[1, 1].set_ylabel('Average Order Value ($)')
axes[1, 1].set_title('Average Order Value by Segment')
axes[1, 1].set_xticks(range(len(avg_values)))
axes[1, 1].set_xticklabels([f'Segment {i}' for i in avg_values.index])

# Add value labels on bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'${height:.0f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 9. Marketing Strategy Recommendations

In [ ]:
# Generate marketing recommendations
print("MARKETING STRATEGY RECOMMENDATIONS")
print("=" * 50)

for cluster, insight in insights.items():
    print(f"\nSEGMENT {cluster}:")
    print(f"Insight: {insight}")
    
    if "VIP" in insight:
        recommendations = [
            "Implement VIP loyalty program with exclusive benefits",
            "Offer early access to new products",
            "Provide premium customer support",
            "Send personalized product recommendations"
        ]
    elif "Loyal" in insight:
        recommendations = [
            "Create targeted cross-selling campaigns",
            "Offer bundle discounts on complementary products",
            "Implement referral reward programs",
            "Send regular newsletters with product updates"
        ]
    elif "At-Risk" in insight:
        recommendations = [
            "Launch win-back email campaigns with special offers",
            "Provide limited-time discount codes",
            "Send surveys to understand why they stopped buying",
            "Retarget with social media advertising"
        ]
    elif "New" in insight or "Low-Value" in insight:
        recommendations = [
            "Create onboarding email series",
            "Offer new customer welcome discounts",
            "Provide product education and tutorials",
            "Implement progressive incentive programs"
        ]
    else:
        recommendations = [
            "Run targeted promotional campaigns",
            "Test different communication channels",
            "Offer seasonal discounts and promotions",
            "Monitor engagement and adjust strategy"
        ]
    
    print("Recommended Actions:")
    for i, rec in enumerate(recommendations, 1):
        print(f"  {i}. {rec}")

## 10. Save Results

In [ ]:
# Prepare final customer segmentation results
customer_features['segment'] = customer_features['cluster']
customer_features['segment_description'] = customer_features['segment'].map(insights)

# Display sample results
print("Sample Customer Segmentation Results:")
print(customer_features[['customer_id', 'recency', 'frequency', 'monetary', 
                        'avg_order_value', 'segment', 'segment_description']].head(10))

# Save to CSV
customer_features.to_csv('../data/customer_segments_notebook.csv', index=False)
print(f"\nResults saved to ../data/customer_segments_notebook.csv")

## Conclusion

This customer segmentation analysis has successfully:

1. **Generated realistic e-commerce transaction data** for demonstration purposes
2. **Applied machine learning techniques** using K-means clustering to identify customer segments
3. **Used RFM analysis** (Recency, Frequency, Monetary) to understand customer behavior
4. **Identified optimal number of clusters** using the elbow method
5. **Generated actionable business insights** for each customer segment
6. **Provided specific marketing recommendations** aligned with business goals
7. **Created comprehensive visualizations** to support data-driven decision making

### Key Takeaways:
- The model effectively separates customers into meaningful segments based on purchasing behavior
- Each segment requires different marketing strategies to maximize engagement and revenue
- Regular re-segmentation with fresh data will help track customer lifecycle changes
- The approach demonstrates fundamental machine learning and data wrangling techniques applicable to real-world e-commerce scenarios

### Next Steps:
1. Implement the recommended marketing strategies for each segment
2. Monitor segment performance and customer migration over time
3. A/B test different approaches within each segment
4. Expand the model with additional features (seasonality, product affinity, geographic data)
5. Integrate with marketing automation platforms for personalized campaigns